In [1]:
import torch
import torch.nn as nn
from torch.optim import Optimizer

In [2]:

class CustomSGDMomentum(Optimizer):
    def __init__(self, params, lr=1e-2, momentum=0.9):
        defaults = dict(lr=lr, momentum=momentum)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr = group['lr']
            beta = group['momentum']

            for p in group['params']:
                if p.grad is None:
                    continue
                
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['velocity'] = torch.zeros_like(p.data)
                v = state['velocity']
                v.mul_(beta).add_(grad)
                p.data.add_(v, alpha=-lr)

        return loss

In [3]:
X = torch.randn(100, 1)
y = 3.5 * X + 1.2 + torch.randn(100, 1) * 0.1

model = nn.Linear(1, 1)
optimizer = CustomSGDMomentum(model.parameters(), lr=0.05, momentum=0.9)
criterion = nn.MSELoss()
losses = []

for epoch in range(40):
    optimizer.zero_grad()
    predictions = model(X)
    loss = criterion(predictions, y)
    losses.append(loss.item())
    loss.backward()
    optimizer.step()

plt.figure(figsize=(7, 4))
plt.plot(losses, color='darkorange', linewidth=2)
plt.title("Training Loss Decreasing Over Time")
plt.xlabel("Optimization Epoch Steps")
plt.ylabel("Mean Squared Error Loss Value")
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined